In [1]:
import requests
import time
from mal_scraper import get_user_anime_list
import logging
import pandas as pd
import numpy as np

In [2]:
logging.getLogger('jikanpy').setLevel(logging.CRITICAL)
logging.getLogger().setLevel(logging.CRITICAL)

In [3]:
delay = 1
maxUserSize = 500

In [4]:
def getRandomUser():
    time.sleep(delay)
    url = "https://api.jikan.moe/v4/random/users"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            return data['data']
        elif response.status_code == 429:
            print("Rate limited! Waiting before retry...")
            time.sleep(0.1)
            return getRandomUser()
        else:
            print(f"Error: {response.status_code}")
            return None
            
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

In [5]:
username = "Exarfate"
print(username)

Exarfate


In [6]:
def getReviewsFromUser(username):
    anime_list = get_user_anime_list(username)
    if anime_list is None: return []
    anime_list = [anime for anime in anime_list if anime['score'] > 0]
    return anime_list

In [7]:
anime_list = getReviewsFromUser(username)
print(anime_list)
print(len(anime_list))

[{'name': 'Code Geass: Hangyaku no Lelouch', 'id_ref': 1575, 'consumption_status': <ConsumptionStatus.completed: 'COMPLETED'>, 'is_rewatch': False, 'score': 10, 'progress': 25, 'start_date': None, 'finished_date': None}, {'name': 'Code Geass: Hangyaku no Lelouch R2', 'id_ref': 2904, 'consumption_status': <ConsumptionStatus.completed: 'COMPLETED'>, 'is_rewatch': False, 'score': 10, 'progress': 25, 'start_date': None, 'finished_date': None}]
2


In [8]:
url = "https://api.jikan.moe/v4/top/anime"
popular_anime_ids = []
for page in range(1, 100):
    params = {
        "filter": "bypopularity",
        "page": page,
    }
    response = requests.get(url, params=params)
    time.sleep(delay)
    data = response.json()
    # print(data)
    animes = data['data']
    popular_anime_ids += [anime['mal_id'] for anime in animes]

print(popular_anime_ids)
print(len(popular_anime_ids))

[16498, 1535, 5114, 30276, 38000, 31964, 11757, 11061, 20, 22319, 40748, 32281, 25777, 9253, 1735, 33486, 21, 35760, 28851, 38524, 19815, 1575, 31240, 23273, 36456, 4224, 32182, 20507, 31043, 40028, 22199, 24833, 23755, 269, 6547, 20583, 37779, 30831, 10620, 21881, 1, 9919, 22535, 199, 30, 33352, 44511, 37450, 38691, 2904, 37999, 28223, 38408, 34572, 27899, 50265, 34134, 14719, 18679, 6702, 37521, 35849, 11111, 40456, 3588, 2001, 35790, 28999, 29803, 13601, 28171, 47778, 37510, 9989, 15809, 32937, 28121, 37430, 42897, 34933, 226, 8074, 10087, 39535, 30654, 14813, 28891, 121, 48583, 30503, 31478, 34599, 38671, 11617, 2167, 40591, 6746, 5081, 431, 12189, 14741, 35507, 26243, 52991, 9756, 42249, 51009, 164, 37520, 20899, 41587, 205, 7054, 33206, 32935, 19, 813, 13759, 48736, 6880, 11771, 36511, 4181, 23283, 39587, 10793, 35120, 4898, 31933, 48561, 33255, 26055, 18897, 18153, 37349, 853, 34577, 32282, 40852, 11741, 17265, 22297, 14227, 37991, 523, 23847, 918, 20785, 36474, 30015, 223, 1789

In [9]:
def get_anime_reviews_jikan(anime_id, page):
    """
    Get reviews for an anime using Jikan API
    """
    url = f"https://api.jikan.moe/v4/anime/{anime_id}/reviews"
    params = {
        "page": page,
    }
    
    response = requests.get(url, params=params)
    time.sleep(delay)
    
    if response.status_code == 200:
        return response.json()
    else:
        return None

In [10]:
number = 1
t0 = time.time()
distinctUsers = set()
to_graph = [0]
for animeId in popular_anime_ids:
    page = 1
    while len(distinctUsers) < maxUserSize:
    # for page in range(1, 10):        
        data = get_anime_reviews_jikan(animeId, page)
        # print(data)
        for elem in data['data']:
            # print(elem)
            for i, review in enumerate(data['data'], 1):
                # print(f"\nReview {i}:")
                # print(f"User: {review['user']['username']}")
                # print(f"Score: {review['score']}")
                distinctUsers.add(review['user']['username'])

        if not data['pagination']['has_next_page']:
            print(f"{number} ended at: {page} with total of {len(distinctUsers)} users after {time.time() - t0} seconds")
            break
        page += 1
    to_graph.append(len(distinctUsers))
    number += 1

In [11]:
usernames = list(distinctUsers)

In [12]:
rows = []
rowsAtLeast5 = []
usernamesAtLeast5 = []
for username in usernames:
    animeList = getReviewsFromUser(username)
    shortedAnimeList = [{
            'id_ref': entry['id_ref'],
            'name': entry['name'],
            'score': entry['score'],
        } 
        for entry in animeList]
    # print(animeList)
    # print(shortedAnimeList)
    # print(len(animeList))
    rows.append(shortedAnimeList)
    if len(shortedAnimeList) > 4:
        rowsAtLeast5.append(shortedAnimeList)
        usernamesAtLeast5.append(username)

In [13]:
with open("usernames.txt", "w") as f:
    for item in distinctUsers:
        f.write(f"{item}\n")

In [14]:
df = pd.DataFrame([
    {f"{item['id_ref']}_{item['name']}": item['score'] for item in userAnime}
    for userAnime in rows
], index=usernames)

df = df.fillna(0)

df

,1_Cowboy Bebop,47_Akira,6746_Durarara!!,10087_Fate/Zero,11741_Fate/Zero 2nd Season,5114_Fullmetal Alchemist: Brotherhood,270_Hellsing,431_Howl no Ugoku Shiro,11061_Hunter x Hunter (2011),14719_JoJo no Kimyou na Bouken (TV),...,59711_Shibou Yuugi de Meshi wo Kuu.,59459_Silent Witch: Chinmoku no Majo no Kakushigoto,54900_Wind Breaker,56009_Yuusha-kei ni Shosu: Choubatsu Yuusha 9004-tai Keimu Kiroku,3269_.hack//G.U. Trilogy,2823_Barbapapa,3446_Biohazard: Degeneration,10917_Pokemon: Pikachu no Fuyuyasumi (2001),5316_Rape! Rape! Rape!,966_Crayon Shin-chan
Bravo_Zulu,10.0,9.0,8.0,9.0,9.0,10.0,9.0,8.0,10.0,9.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
shadovvvvalker,10.0,0.0,0.0,9.0,9.0,8.0,8.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Beastm0de23,8.0,8.0,0.0,8.0,8.0,10.0,0.0,8.0,9.0,8.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Korukasu,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
deathtousall173,9.0,0.0,8.0,0.0,0.0,9.0,0.0,6.0,8.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SaintSinner_,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
themegamancave,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Skittles04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
anarrowstrail,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [15]:
df2 = pd.DataFrame([
    {f"{item['id_ref']}_{item['name']}": item['score'] for item in userAnime}
    for userAnime in rowsAtLeast5
], index=usernamesAtLeast5)

df2 = df2.fillna(0)

df2

,1_Cowboy Bebop,47_Akira,6746_Durarara!!,10087_Fate/Zero,11741_Fate/Zero 2nd Season,5114_Fullmetal Alchemist: Brotherhood,270_Hellsing,431_Howl no Ugoku Shiro,11061_Hunter x Hunter (2011),14719_JoJo no Kimyou na Bouken (TV),...,59711_Shibou Yuugi de Meshi wo Kuu.,59459_Silent Witch: Chinmoku no Majo no Kakushigoto,54900_Wind Breaker,56009_Yuusha-kei ni Shosu: Choubatsu Yuusha 9004-tai Keimu Kiroku,3269_.hack//G.U. Trilogy,2823_Barbapapa,3446_Biohazard: Degeneration,10917_Pokemon: Pikachu no Fuyuyasumi (2001),5316_Rape! Rape! Rape!,966_Crayon Shin-chan
Bravo_Zulu,10.0,9.0,8.0,9.0,9.0,10.0,9.0,8.0,10.0,9.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
shadovvvvalker,10.0,0.0,0.0,9.0,9.0,8.0,8.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Beastm0de23,8.0,8.0,0.0,8.0,8.0,10.0,0.0,8.0,9.0,8.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Korukasu,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
deathtousall173,9.0,0.0,8.0,0.0,0.0,9.0,0.0,6.0,8.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
yhyhyhkt,10.0,0.0,0.0,0.0,0.0,9.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AnimeCrew,0.0,0.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Katanadude,10.0,0.0,0.0,8.0,9.0,9.0,6.0,0.0,1.0,7.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ghier,0.0,7.0,7.0,9.0,9.0,8.0,0.0,7.0,8.0,0.0,...,5.0,5.0,4.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0


In [16]:
df.to_csv('anime_df.csv')  # index=False removes row numbers
df

,1_Cowboy Bebop,47_Akira,6746_Durarara!!,10087_Fate/Zero,11741_Fate/Zero 2nd Season,5114_Fullmetal Alchemist: Brotherhood,270_Hellsing,431_Howl no Ugoku Shiro,11061_Hunter x Hunter (2011),14719_JoJo no Kimyou na Bouken (TV),...,59711_Shibou Yuugi de Meshi wo Kuu.,59459_Silent Witch: Chinmoku no Majo no Kakushigoto,54900_Wind Breaker,56009_Yuusha-kei ni Shosu: Choubatsu Yuusha 9004-tai Keimu Kiroku,3269_.hack//G.U. Trilogy,2823_Barbapapa,3446_Biohazard: Degeneration,10917_Pokemon: Pikachu no Fuyuyasumi (2001),5316_Rape! Rape! Rape!,966_Crayon Shin-chan
Bravo_Zulu,10.0,9.0,8.0,9.0,9.0,10.0,9.0,8.0,10.0,9.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
shadovvvvalker,10.0,0.0,0.0,9.0,9.0,8.0,8.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Beastm0de23,8.0,8.0,0.0,8.0,8.0,10.0,0.0,8.0,9.0,8.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Korukasu,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
deathtousall173,9.0,0.0,8.0,0.0,0.0,9.0,0.0,6.0,8.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SaintSinner_,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
themegamancave,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Skittles04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
anarrowstrail,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [17]:
print(len(usernames))

500
